In [1]:
!pip install ultralytics wandb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.8 MB/s eta 0:00:00a 0:00:01


In [2]:
import wandb
from kaggle_secrets import UserSecretsClient
wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)


def wandb_epoch_end(trainer):
    """Log one epoch of Ultralytics metrics to W&B."""

    log = {"epoch": trainer.epoch + 1}

    # Training losses
    if hasattr(trainer, "tloss"):
        names = ["train/box_loss", "train/cls_loss", "train/dfl_loss"]
        for i, v in enumerate(trainer.tloss):
            if i < len(names):
                log[names[i]] = float(v)

    # Validation metrics
    if hasattr(trainer, "metrics"):
        try:
            for k, v in trainer.metrics.items():
                log[k] = float(v)
        except Exception:
            pass

    # Learning rates
    if hasattr(trainer, "lr"):
        for k, v in trainer.lr.items():
            log[f"lr/{k}"] = float(v)

    wandb.log(log)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vantu12772 (vantu12772-fpt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
from pathlib import Path
import yaml
import torch
import types
import shutil
from ultralytics import YOLO

DATASET = Path("/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive")
IMAGES  = DATASET / "images"
LABELS  = DATASET / "labels"
SPLIT   = DATASET / "split_dataset"
CLASSES = DATASET / "classes_en.txt"

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True, parents=True)

WANDB_PROJECT = "vnts-traffic-signs"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
def write_paths(split_txt, out_txt):
    with open(split_txt, encoding="utf-8") as f:
        names = [l.strip() for l in f if l.strip()]
    paths = [str(IMAGES / name) for name in names]
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(paths))
    print(f"Written {len(paths)} paths -> {out_txt}")

write_paths(SPLIT / "train_files.txt", WORK / "train_paths.txt")
write_paths(SPLIT / "test_files.txt",  WORK / "val_paths.txt")

with open(CLASSES, encoding="utf-8") as f:
    class_names = [l.strip() for l in f if l.strip()]
n_classes = len(class_names)

data = {
    "path":  str(WORK),
    "train": "train_paths.txt",
    "val":   "val_paths.txt",
    "nc":    n_classes,
    "names": class_names,
}
yaml_path = WORK / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data, f, allow_unicode=True, sort_keys=False)

print(f"Classes: {n_classes}, yaml -> {yaml_path}")

Written 2552 paths -> /kaggle/working/train_paths.txt
Written 639 paths -> /kaggle/working/val_paths.txt
Classes: 52, yaml -> /kaggle/working/data.yaml


In [ ]:
run = wandb.init(
    project="vnts-hls",
    name="Model_B",
    reinit=True,

    config={
        "model": "YOLO26n",
        "loss": "Default YOLO",
        "epochs": 100,
        "batch": 32,
        "imgsz": 640,
    }
)

model_B = YOLO("yolo26n.pt")
model_B.add_callback(
    "on_fit_epoch_end",
    wandb_epoch_end
)
results_B = model_B.train(
    data=str(yaml_path), epochs=100, batch=32, imgsz=640,
    lr0=0.01, lrf=0.01, seed=0,
    project="/kaggle/working/runs", name="model_B",
)

wandb.finish()

SETUP HLS

In [ ]:
from ultralytics.utils.loss import v8DetectionLoss, E2ELoss
from ultralytics.utils.tal import make_anchors

CATEGORY_MAPPING = {
    2:0, 7:0, 10:0, 14:0, 16:0, 17:0, 18:0, 19:0, 20:0,
    23:0, 24:0, 29:0, 32:0, 34:0, 35:0, 36:0, 38:0, 39:0,
    40:0, 41:0, 46:0, 47:0, 49:0,
    0:1, 1:1, 4:1, 5:1, 6:1, 13:1, 15:1, 21:1, 22:1,
    26:1, 27:1, 33:1, 37:1, 43:1, 44:1, 48:1, 50:1, 51:1,
    3:2, 9:2, 12:2, 42:2,
    8:3, 11:3, 31:3, 45:3,
    25:4, 28:4, 30:4,
}

def build_smoothing_matrix(mapping, num_classes=52, alpha=0.05, beta=0.10):
    sibling_sets = {}
    for cls_id, cat_id in mapping.items():
        sibling_sets.setdefault(cat_id, []).append(cls_id)
    M = torch.zeros((num_classes, num_classes))
    for true_cls in range(num_classes):
        cat_id   = mapping[true_cls]
        siblings = sibling_sets[cat_id]
        S = len(siblings)
        O = num_classes - S
        for k in range(num_classes):
            if k == true_cls:
                M[true_cls, k] = 1.0 - alpha - beta
            elif k in siblings:
                M[true_cls, k] = beta / (S - 1) if S > 1 else 0.0
            else:
                M[true_cls, k] = alpha / O if O > 0 else 0.0
    return M

class HierarchicalDetectionLoss(v8DetectionLoss):
    def __init__(self, model, mapping=CATEGORY_MAPPING,
                 alpha=0.05, beta=0.10, tal_topk=10, tal_topk2=None):
        super().__init__(model, tal_topk=tal_topk, tal_topk2=tal_topk2)
        self.smoothing_matrix = build_smoothing_matrix(
            mapping, self.nc, alpha, beta
        ).to(self.device)

        self._debug_once = False

    def get_assigned_targets_and_loss(self, preds, batch):
        loss = torch.zeros(3, device=self.device)
        pred_distri, pred_scores = (
            preds["boxes"].permute(0, 2, 1).contiguous(),
            preds["scores"].permute(0, 2, 1).contiguous(),
        )
        anchor_points, stride_tensor = make_anchors(preds["feats"], self.stride, 0.5)
        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = (torch.tensor(preds["feats"][0].shape[2:], device=self.device, dtype=dtype)
                 * self.stride[0])

        targets = torch.cat((batch["batch_idx"].view(-1, 1),
                              batch["cls"].view(-1, 1),
                              batch["bboxes"]), 1)
        targets = self.preprocess(targets.to(self.device), batch_size,
                                   scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0.0)
        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)

        (_, target_bboxes, target_scores, fg_mask, target_gt_idx) = self.assigner(
            pred_scores.detach().sigmoid(),
            (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor,
            gt_labels, gt_bboxes, mask_gt,
        )

        if fg_mask.any():

            hard_labels = target_scores.argmax(dim=-1)
            smooth_targets = self.smoothing_matrix[hard_labels]

            if not self._debug_once:

                batch_idx, anchor_idx = fg_mask.nonzero()[0]

                print("\n========== HLS DEBUG ==========")

                print(
                    f"Ground-truth class: "
                    f"{hard_labels[batch_idx, anchor_idx].item()}")

                print("\nOriginal target (first 10):")
                print(target_scores[batch_idx, anchor_idx][:10])

                print("\nSmoothed target (first 10):")
                print(smooth_targets[batch_idx, anchor_idx][:10])

                print(
                    f"\nOriginal max : "
                    f"{target_scores[batch_idx, anchor_idx].max().item():.4f}")

                print(
                    f"Smoothed max : "
                    f"{smooth_targets[batch_idx, anchor_idx].max().item():.4f}")

                print(
                    f"Row sum : "
                    f"{smooth_targets[batch_idx, anchor_idx].sum().item():.4f}")

                print("===============================\n")

                self._debug_once = True

            # Replace one-hot labels with hierarchical smoothed labels
            target_scores = torch.where(
            fg_mask.unsqueeze(-1),
            smooth_targets,
            target_scores,
                            )

        target_scores_sum = max(target_scores.sum(), 1)
        bce_loss = self.bce(pred_scores, target_scores.to(dtype))
        if self.class_weights is not None:
            bce_loss *= self.class_weights
        loss[1] = bce_loss.sum() / target_scores_sum

        if fg_mask.sum():
            loss[0], loss[2] = self.bbox_loss(
                pred_distri, pred_bboxes, anchor_points,
                target_bboxes / stride_tensor, target_scores,
                target_scores_sum, fg_mask, imgsz, stride_tensor,
            )

        loss[0] *= self.hyp.box
        loss[1] *= self.hyp.cls
        loss[2] *= self.hyp.dfl
        return ((fg_mask, target_gt_idx, target_bboxes, anchor_points, stride_tensor),
                loss, loss.detach())

class HierarchicalE2ELoss(E2ELoss):
    def __init__(self, model, mapping=CATEGORY_MAPPING, alpha=0.05, beta=0.10):
        super().__init__(model)
        self.one2many = HierarchicalDetectionLoss(model, mapping, alpha, beta, tal_topk=10)
        self.one2one  = HierarchicalDetectionLoss(model, mapping, alpha, beta, tal_topk=7, tal_topk2=1)

def make_hls_patch(alpha, beta):
    def patch_model(trainer):
        model = trainer.model
        if model is None:
            return
        def patched_init_criterion(self):
            return HierarchicalE2ELoss(self, mapping=CATEGORY_MAPPING, alpha=alpha, beta=beta)
        model.init_criterion = types.MethodType(patched_init_criterion, model)
        model.criterion = model.init_criterion()
        print(f"HLS criterion active: alpha={alpha}, beta={beta}")
    return patch_model
print("Finished HLS")

MODEL A TRAINING AND FINE TUNING BELOW:

In [ ]:



run = wandb.init(
    project="vnts-hls",
    name="Model_A",
    reinit=True,

    config={
        "model": "YOLO26n",
        "loss": "HLS",
        "alpha": 0.05,
        "beta": 0.10,
        "epochs": 100,
        "batch": 32,
        "imgsz": 640,
    }
)


model_A = YOLO("yolo26n.pt")


model_A.add_callback("on_train_start", make_hls_patch(alpha=0.05, beta=0.10))

model_A.add_callback("on_fit_epoch_end", wandb_epoch_end)

print("🚀 Launching Model A — Hierarchical Label Smoothing")

results_A = model_A.train(
    data            = str(yaml_path),
    epochs          = 100,
    batch           = 32,
    imgsz           = 640,
    lr0             = 0.01,
    lrf             = 0.01,
    seed            = 0,
    project         = "/kaggle/working/runs",
    name            = "model_A",
    label_smoothing = 0.0
)

wandb.finish()

In [ ]:
run = wandb.init(
    project="vnts-hls",
    name="Model_A_HLS_v2",
    reinit=True,

    config={
        "model": "YOLO26n",
        "loss": "HLS Wide Gap",
        "alpha": 0.01,
        "beta": 0.20,
        "epochs": 100,
        "batch": 32,
        "imgsz": 640,
    }
)

model_v2 = YOLO("yolo26n.pt")


model_v2.add_callback("on_train_start", make_hls_patch(alpha=0.01, beta=0.20))

model_v2.add_callback("on_fit_epoch_end", wandb_epoch_end)


results_v2 = model_v2.train(
    data=str(yaml_path), epochs=100, batch=32, imgsz=640,
    lr0=0.01, lrf=0.01, seed=0, label_smoothing=0.0,
    project="/kaggle/working/runs", name="A_v2_widegap",
)

wandb.finish()

In [ ]:
run = wandb.init(
    project="vnts-hls",
    name="Model_A_HLS_v3",
    reinit=True,

    config={
        "model": "YOLO26n",
        "loss": "HLS + cls=1.5",
        "alpha": 0.05,
        "beta": 0.10,
        "epochs": 100,
        "batch": 32,
        "imgsz": 640,
        "cls":1.5,
    }
)
model_v3 = YOLO("yolo26n.pt")
model_v3.add_callback("on_train_start", make_hls_patch(alpha=0.05, beta=0.10))

model_v3.add_callback("on_fit_epoch_end", wandb_epoch_end)


results_v3 = model_v3.train(
    data=str(yaml_path), epochs=100, batch=32, imgsz=640,
    lr0=0.01, lrf=0.01, seed=0, label_smoothing=0.0,
    cls=1.5,
    project="/kaggle/working/runs", name="A_v3_highcls",
)

wandb.finish()

In [ ]:
run = wandb.init(
    project="vnts-hls",
    name="Model_A_HLS_v4",
    reinit=True,

    config={
        "model": "YOLO26n",
        "loss": "HLS Wide Gap + cls=1.5",
        "alpha": 0.01,
        "beta": 0.20,
        "epochs": 100,
        "batch": 32,
        "imgsz": 640,
        "cls":1.5,
    }
)
model_v4 = YOLO("yolo26n.pt")
model_v4.add_callback("on_train_start", make_hls_patch(alpha=0.01, beta=0.20))


model_v4.add_callback("on_fit_epoch_end", wandb_epoch_end)

results_v4 = model_v4.train(
    data=str(yaml_path), epochs=100, batch=32, imgsz=640,
    lr0=0.01, lrf=0.01, seed=0, label_smoothing=0.0,
    cls=1.5,
    project="/kaggle/working/runs", name="A_v4_combined",
)


wandb.finish()

In [ ]:
for name in ["model_B", "model_A", "A_v2_widegap", "A_v3_highcls", "A_v4_combined", "model_C"]:
    run_dir = sorted((WORK / "runs" / name).parent.glob(f"{name}*"))
    if not run_dir:
        run_dir = list((WORK).rglob(name))
    if run_dir:
        shutil.make_archive(str(WORK / f"{name}_outputs"), "zip", str(run_dir[-1]))
        print(f"Saved {name}_outputs.zip")

In [7]:
import sys

# List of all third-party libraries used across your 3 notebooks
libraries = [
    "wandb",
    "kaggle_secrets",
    "torch",
    "ultralytics",
    "numpy",
    "pandas",
    "matplotlib",
    "scipy",
    "seaborn",
    "yaml"  # Imported as 'yaml', but packaged as 'pyyaml'
]

print("=== Copy these versions for your requirements.txt ===\n")

for lib in libraries:
    try:
        # Dynamically import the module
        module = __import__(lib)
        
        # Determine the correct requirements.txt package name
        req_name = "pyyaml" if lib == "yaml" else lib
        
        # Grab the version attribute
        if hasattr(module, "__version__"):
            print(f"{req_name}=={module.__version__}")
        elif hasattr(module, "version"):
            print(f"{req_name}=={module.version}")
        else:
            print(f"{req_name}==[Version not found dynamically, check manually]")
            
    except ImportError:
        # Handle cases where a library might not be installed in the current notebook environment
        print(f"{lib}==[Not installed in this environment]")

=== Copy these versions for your requirements.txt ===

wandb==0.26.1
kaggle_secrets==[Version not found dynamically, check manually]
torch==2.10.0+cu128
ultralytics==8.4.92
numpy==2.0.2
pandas==2.3.3
matplotlib==3.10.0
scipy==1.16.3
seaborn==0.13.2
pyyaml==6.0.3


In [13]:
run = wandb.init(
    project="vnts-hls",
    name="Model_C_P2_SmallObject",
    reinit=True,

    config={
        "model": "YOLO26n-P2",
        "architecture": "yolo26n-p2.yaml (P2/4 small-object head)",
        "pretrained": "backbone weights from yolo26n.pt (P2 layer random init)",
        "epochs": 100,
        "batch": 32,
        "imgsz": 640,
    }
)

model_p2 = YOLO("yolo26n-p2.yaml").load("yolo26n.pt")
model_p2.add_callback("on_fit_epoch_end", wandb_epoch_end)

results_p2 = model_p2.train(
    data=str(yaml_path), epochs=100, batch=32, imgsz=640,
    lr0=0.01, lrf=0.01, seed=0,
    project="/kaggle/working/runs", name="model_P2",
)

wandb.finish()

Transferred 360/902 items from pretrained weights
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-p2.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=model_P2, nbs=64, nms=False, opse

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█
lr/lr/pg0,████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▁
lr/lr/pg1,███████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▁▁▁
lr/lr/pg2,▆████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁
metrics/mAP50(B),▁▁▂▂▂▃▃▄▄▄▄▅▅▅▆▇▇▇▇▇▇▇▇█▇███████████████
metrics/mAP50-95(B),▁▁▁▂▂▂▃▃▃▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇████████████████
metrics/precision(B),▂▁▂▁▂▄▃▄▅▄▆▄▄▆▆▄▅▅▆▇▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇█▇███
metrics/recall(B),▁▁▁▂▂▂▃▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇███████▇█▇██
train/box_loss,█▇▆▅▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁
train/cls_loss,█▇▆▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+4,...


In [14]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working")

run_dir = sorted((WORK / "runs").glob("model_P2*"))
if run_dir:
    latest_run = run_dir[-1]
    shutil.make_archive(str(WORK / "model_P2_outputs"), "zip", str(latest_run))
    print(f"Saved -> model_P2_outputs.zip")
    print(f"From  -> {latest_run}")
else:
    print("WARNING: model_P2 run folder not found — check the exact folder name below:")
    for p in (WORK / "runs").glob("*"):
        print(f"  {p}")

Saved -> model_P2_outputs.zip
From  -> /kaggle/working/runs/model_P2


In [15]:
from IPython.display import FileLink

# Replace with your actual file name
FileLink('/kaggle/working/wandb')
FileLink('/kaggle/working/runs/model_P2')

ValueError: Cannot display a directory using FileLink. Use FileLinks to display '/kaggle/working/wandb'.